# N-Gram Analysis Annotated
This notebook aims at investigating the annotations of n-grams.

In [9]:
import os
import pandas as pd
from tqdm import tqdm
import json

df_ngrams_annotated = pd.read_csv(
    os.path.join("data", "tags", "top1000ngrams_annotated.csv"))

df_ngrams_annotated[["category", "ngram"]].groupby("category").agg(list)


,ngram
category,
-,"[youtube, https, com, auto, generated, by_yout..."
artist type - band,"[band, band]"
artist type - dj,[dj]
artist type - orchestra,"[orchestra, his_orchestra, and_his_orchestra]"
artist type - quartet,[quartet]
artist type - singer,[singer]
artist type - trio,[trio]
auto generated,"[auto_generated, generated_by, generated_by_yo..."
composition - focus on lyrics,"[lyricist, lyrics, letra]"


### Cleaning up tags
We just keep annotations with `sonority`, `situational context` or `composition`. 


In [10]:
keywords = (
    "artist type", "auto generated", "composition", "genre", 
    "instrument vocals", "segment", "situational context", "sonority"
    )

def keep_condition(s):
    if not isinstance(s, str):
        return False
    s_norm = s.strip().lower()
    return any(k in s_norm for k in keywords)

# show matching category names (cleaned)
matching_categories = sorted({cat.strip() for cat in df_ngrams_annotated["category"].unique() if keep_condition(cat)})

# filter dataframe to keep only rows whose category matches the condition
df_tags = df_ngrams_annotated[df_ngrams_annotated["category"].isin(matching_categories)].copy()
df_tags["category"] = df_tags["category"].replace({
    "artist type": "instruments groups",
    "instrument vocals": "instruments groups",
})


df_tags_grouped = df_tags[["category", "ngram"]].groupby("category", as_index=False).agg(list)#.to_dict(orient="index")
df_tags_grouped


,category,ngram
0,artist type - band,"[band, band]"
1,artist type - dj,[dj]
2,artist type - orchestra,"[orchestra, his_orchestra, and_his_orchestra]"
3,artist type - quartet,[quartet]
4,artist type - singer,[singer]
5,artist type - trio,[trio]
6,auto generated,"[auto_generated, generated_by, generated_by_yo..."
7,composition - focus on lyrics,"[lyricist, lyrics, letra]"
8,genre - blues,"[blues, blue]"
9,genre - classical,"[classic, classics]"


## Add a few manually found cues

In [11]:
import yaml

tag_map = dict(zip(df_tags_grouped["category"], df_tags_grouped["ngram"]))

# add manually
tag_map["artist type - band"] = ["band"]
tag_map["artist type - big band"] = ["big band"]
tag_map["segment - verse"] = ["verse"]
tag_map["segment - intro"] = ["intro"]
tag_map["segment - outro"] = ["outro"]
tag_map['situational context - cover'].append("interpretation")
tag_map['situational context - reaction'].append("first_time")
tag_map['situational context - reaction'].append("analysis")
tag_map['situational context - reaction'].append("reviews")
tag_map['situational context - reaction'].append("review")
tag_map["sonority - electric"] = ["electric"]
tag_map['sonority - hq'].append("hq")
tag_map['sonority - hq'].append("high_quality")
tag_map['sonority - hq'].append("high_definition")
tag_map['sonority - instrumental'].append("no_vocals")

# sort each value list, then sort dict by keys
tag_map = {k: sorted(v) for k, v in sorted(tag_map.items())}

out_dir = os.path.join("data", "tags")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "tags.yaml")

with open(out_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(tag_map, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Saved tag_map to {out_path}")


Saved tag_map to data/tags/tags.yaml


### Translate to frequent European languages
We use *GPT-5* (via the GUI) to translate the cues into other languages.


##### Prompt
*You are an expert translator for the domain of online music videos. You are given a YAML file to translate. The file maps categories which are associated with properties of an audio signal of music to cues which can hint these properties when mentioned in the video metadata of online videos. I want to transform this map into a multilingual one with the languages English, Spanish, French, Portuguese, German and Italian. Please reorganize it and arrange lists per language in each of the categories. Put the existing cues in the list of their corresponding language and translate into all the other languages but constrain the translation to the domain of music on online video platforms. Furthermore, please include all conjugations and plurals if applicable.* 

